# Baseline: TF-IDF + Logistic Regression

Our first baseline for toxic comment detection (originally run on Kaggle, so some cells use `/kaggle/working` paths).
It trains the model on the 25k training sample, evaluates it on the validation set and looks at errors by confidence group.

The main results for Deliverable 1 are in `active_learning/active_learning_experiments.ipynb`.

In [ ]:
from datasets import load_dataset

ds = load_dataset("google/civil_comments")

In [ ]:
print(ds)
print(ds["train"][0])

In [ ]:
ds.save_to_disk("/kaggle/working/civil_comments")

In [ ]:
import os

print(os.path.isdir)

In [ ]:
print(ds)

In [ ]:
import os

print("Current directory:", os.getcwd())
print("Working directory exists:", os.path.isdir("/kaggle/working"))
print("Working directory contents:", os.listdir("/kaggle/working"))

print("Dataset variable exists:", "ds" in globals())
if "ds" in globals():
    print(ds)

In [ ]:
from datasets import load_from_disk
import os

path = "/kaggle/working/civil_comments"

print("Dataset folder exists:", os.path.isdir(path))

if os.path.isdir(path):
    ds = load_from_disk(path)
    print(ds)
else:
    print("The dataset is not at this path. Check the earlier save output.")

In [ ]:
# Sample from each original partition
train_ds = ds["train"].train_test_split(train_size=25_000, seed=42)["train"]
val_ds = ds["validation"].train_test_split(train_size=5_000, seed=42)["train"]
test_ds = ds["test"].train_test_split(train_size=5_000, seed=42)["train"]

# Convert to Pandas
train_df = train_ds.to_pandas()
val_df = val_ds.to_pandas()
test_df = test_ds.to_pandas()

train_df.head()

In [ ]:
# Define binary toxicity labels
for df in [train_df, val_df, test_df]:
    df["label"] = (df["toxicity"] >= 0.5).astype(int)

train_df[["text", "toxicity", "label"]].head()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

RANDOM_STATE = 42

# Text representation
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_features=20_000,
    sublinear_tf=True
)

X_train = vectorizer.fit_transform(train_df["text"])
X_val = vectorizer.transform(val_df["text"])
y_train = train_df["label"]
y_val = val_df["label"]

# Train classifier
model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=RANDOM_STATE
)

model.fit(X_train, y_train)

# Predictions on validation data
y_proba = model.predict_proba(X_val)[:, 1]
y_pred = model.predict(X_val)

print("Training completed.")
print("Number of features:", X_train.shape[1])

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

print (accuracy_score(y_val, y_pred))
print (f1_score(y_val, y_pred, average="macro"))

In [ ]:
print("Training data distribution:")
print(train_df["label"].value_counts())

print("\nPercentage distribution:")
print(train_df["label"].value_counts(normalize=True) * 100)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    y_val,
    y_pred,
    target_names=["Non-toxic", "Toxic"],
    digits=4
))

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm = confusion_matrix(y_val, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Non-toxic", "Toxic"]
)

disp.plot(cmap="Blues", values_format="d")
plt.title("Baseline Confusion Matrix")
plt.show()

print(cm)

In [ ]:
# Create a DataFrame containing model predictions

results_df = val_df.copy()

results_df["predicted_label"] = y_pred
results_df["predicted_probability"] = y_proba

# Identify prediction errors

false_positives = results_df[
    (results_df["label"] == 0) &
    (results_df["predicted_label"] == 1)
]

false_negatives = results_df[
    (results_df["label"] == 1) &
    (results_df["predicted_label"] == 0)
]

print("False Positives:", len(false_positives))
print("False Negatives:", len(false_negatives))

In [ ]:
false_positives[
    ["text", "toxicity", "predicted_probability"]
].head(10)

In [ ]:
false_negatives[
    ["text", "toxicity", "predicted_probability"]
].head(10)

In [ ]:
import pandas as pd

pd.set_option("display.max_colwidth", None)

for index, row in false_positives.head(10).iterrows():

    print("\n" + "=" * 80)
    print("FALSE POSITIVE")
    print("Actual toxicity:", row["toxicity"])
    print("Predicted probability:", row["predicted_probability"])
    print("Comment:")
    print(row["text"])

In [ ]:
for index, row in false_negatives.head(10).iterrows():

    print("\n" + "=" * 80)
    print("FALSE NEGATIVE")
    print("Actual toxicity:", row["toxicity"])
    print("Predicted probability:", row["predicted_probability"])
    print("Comment:")
    print(row["text"])

In [ ]:
y_proba = model.predict_proba(X_val)[:, 1]

In [ ]:
print("FALSE POSITIVE PROBABILITY STATISTICS")
print(false_positives["predicted_probability"].describe())

print("\nFALSE NEGATIVE PROBABILITY STATISTICS")
print(false_negatives["predicted_probability"].describe())

In [ ]:
TN, FP, FN, TP = cm.ravel()

false_positive_rate = FP / (FP + TN)
false_negative_rate = FN / (FN + TP)

print("False Positive Rate:",
      round(false_positive_rate * 100, 2), "%")

print("False Negative Rate:",
      round(false_negative_rate * 100, 2), "%")

In [ ]:
import pandas as pd

results_df["confidence_group"] = pd.cut(
    results_df["predicted_probability"],
    bins=[0, 0.30, 0.70, 1.0],
    labels=[
        "High-confidence non-toxic",
        "Uncertain",
        "High-confidence toxic"
    ],
    include_lowest=True
)

print("Number of comments in each group:")

print(results_df["confidence_group"].value_counts())

In [ ]:
results_df["correct_prediction"] = (
    results_df["label"] == results_df["predicted_label"]
)

summary = results_df.groupby(
    "confidence_group",
    observed=False
).agg(
    total_comments=("label", "count"),
    correct_predictions=("correct_prediction", "sum")
)

summary["incorrect_predictions"] = (
    summary["total_comments"] -
    summary["correct_predictions"]
)

summary["error_percentage"] = (
    summary["incorrect_predictions"] /
    summary["total_comments"] * 100
)

print(summary)

In [ ]:
# Analyse errors by confidence group

group_analysis = results_df.groupby(
    "confidence_group",
    observed=False
).apply(
    lambda group: pd.Series({
        "Total": len(group),

        "True Positives": (
            (group["label"] == 1) &
            (group["predicted_label"] == 1)
        ).sum(),

        "True Negatives": (
            (group["label"] == 0) &
            (group["predicted_label"] == 0)
        ).sum(),

        "False Positives": (
            (group["label"] == 0) &
            (group["predicted_label"] == 1)
        ).sum(),

        "False Negatives": (
            (group["label"] == 1) &
            (group["predicted_label"] == 0)
        ).sum()
    })
)

print(group_analysis)

In [ ]:
# Simulate ideal human correction of uncertain predictions

results_df["human_assisted_prediction"] = (
    results_df["predicted_label"]
)

# Replace AI predictions with dataset labels
# for comments assigned to human review

review_mask = (
    results_df["confidence_group"] == "Uncertain"
)

results_df.loc[
    review_mask,
    "human_assisted_prediction"
] = results_df.loc[
    review_mask,
    "label"
]

# Evaluate AI-only and idealised human-assisted accuracy

from sklearn.metrics import accuracy_score

ai_accuracy = accuracy_score(
    results_df["label"],
    results_df["predicted_label"]
)

human_assisted_accuracy = accuracy_score(
    results_df["label"],
    results_df["human_assisted_prediction"]
)

print("AI-only accuracy:", round(ai_accuracy * 100, 2), "%")

print(
    "Idealised human-assisted accuracy:",
    round(human_assisted_accuracy * 100, 2),
    "%"
)

print(
    "Comments sent for human review:",
    review_mask.sum()
)

print(
    "Human review workload:",
    round(review_mask.mean() * 100, 2),
    "%"
)